# 5. Classificare e segmentare un PDF multi-documento

Usa le categorie in [modelli/classificatore_documenti.json](../modelli/classificatore_documenti.json). Risultati attesi: hotel sulle pagine 1-2, ristorante a pagina 3, altro a pagina 4. L'ID del classificatore arriva da `.env`; CU restituisce gli intervalli, mostriamo solo categorie e intervalli, senza creare file.

Documento: [esercizio5.pdf](../file/esercizio5.pdf).

Soluzione completa: esegui le celle in ordine con il kernel del venv. Ogni notebook parte da zero: nessun `main()`, `%run` o stato condiviso con gli altri esercizi.

[Traccia da svolgere](../esercizi/05-classificazione.ipynb).

## 1. Import e configurazione

Esegui dalla cartella `soluzioni`. `load_dotenv(r"..\.env", override=True)` carica il file noto e applica i suoi valori. I risultati vengono soltanto stampati.

In [1]:
import json
import os
from pathlib import Path
from azure.ai.contentunderstanding import ContentUnderstandingClient
from azure.ai.contentunderstanding.models import ContentAnalyzer
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv

In [2]:
load_dotenv(r"..\.env", override=True)
DOCUMENTO = Path(r"..\file\esercizio5.pdf")
document_bytes = DOCUMENTO.read_bytes()

# Legge la definizione JSON, non il risultato di un'analisi.
definizione = json.loads(
    Path(r"..\modelli\classificatore_documenti.json").read_text(encoding="utf-8")
)
classifier_id = os.environ["AZURE_CONTENTUNDERSTANDING_CUSTOM_CLASSIFIER_ANALYZER_ID"]

## 2. Client ed esecuzione

`ContentAnalyzer(definizione)` usa direttamente il JSON, senza verifiche sui deployment. `begin_create_analyzer` riceve ID, definizione e `allow_replace=True`; `begin_analyze_binary` riceve ID e byte PDF. `result()` attende il completamento.

`with` chiude client e credenziale; `finally` elimina l'analyzer dopo l'analisi, anche se fallisce. Usa solo ID dedicati al laboratorio: `allow_replace=True` sostituisce l'analyzer esistente. Non eseguire due copie dello stesso notebook contemporaneamente. Un riavvio forzato del kernel puo impedire la pulizia.

In [ ]:
# Crea client e credenziale; with li chiude anche in caso di errore.
with DefaultAzureCredential() as credential, ContentUnderstandingClient(
    endpoint=os.environ["AZURE_CONTENTUNDERSTANDING_ENDPOINT"], credential=credential,
) as client:
    # Crea l'analyzer dal JSON; allow_replace aggiorna l'ID del laboratorio.
    client.begin_create_analyzer(
        analyzer_id="classificatore_documenti", resource=ContentAnalyzer(definizione), allow_replace=True,
    ).result()
    try:
        # Analizza i byte del PDF con l'analyzer indicato e attende il risultato.
        result = client.begin_analyze_binary(
            analyzer_id="classificatore_documenti", binary_input=document_bytes,
        ).result()
    finally:
        # Elimina l'analyzer temporaneo identificato da analyzer_id.
        client.delete_analyzer(analyzer_id="classificatore_documenti")

## 3. Osserva il risultato

Mostra il risultato della chiamata Azure. `as_dict()` converte gli oggetti SDK in dizionari; `json.dumps(..., ensure_ascii=False, indent=2)` produce JSON leggibile. `print` mostra il risultato senza salvarlo su disco.

In [4]:
# as_dict espone categoria, pagine e confidence restituite dal servizio.
segmenti = [
    {
        "categoria": segment["category"],
        "pagina_iniziale": segment["startPageNumber"],
        "pagina_finale": segment["endPageNumber"],
        "confidence": segment["confidence"],
    }
    for content in result.as_dict()["contents"]
    for segment in content["segments"]
]
print(json.dumps(segmenti, ensure_ascii=False, indent=2))

[
  {
    "categoria": "hotel_receipt",
    "pagina_iniziale": 1,
    "pagina_finale": 2,
    "confidence": 0.98
  },
  {
    "categoria": "restaurant_receipt",
    "pagina_iniziale": 3,
    "pagina_finale": 3,
    "confidence": 0.99
  },
  {
    "categoria": "other",
    "pagina_iniziale": 4,
    "pagina_finale": 4,
    "confidence": 0.99
  }
]
